This notebook covers how a lead connects to the boat it is about, and how well those join keys resolve into the listing and customer tables. A join rate here means the share of usable leads that find a match through that key.

In [ ]:
WITH base AS (
  SELECT LEAD_ID, NULLIF(TRIM(GA_CLIENT_ID),'') AS gacid, LISTING_ID
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
)
SELECT
  COUNT(*)                                       AS total_leads,
  COUNT(base.gacid)                              AS has_ga_client_id,
  COUNT(base.LISTING_ID)                         AS has_listing_id,
  COUNT(s.LISTING_ID)                            AS listing_resolves_in_snapshot,
  ROUND(COUNT(base.gacid)*100.0/COUNT(*),1)      AS pct_has_ga_client_id,
  ROUND(COUNT(base.LISTING_ID)*100.0/COUNT(*),1) AS pct_has_listing_id,
  ROUND(COUNT(s.LISTING_ID)*100.0/COUNT(*),1)    AS pct_listing_resolves
FROM base
LEFT JOIN AWS_DP_PROD.LISTINGS.LISTING_CURRENT_SNAPSHOT s
  ON s.LISTING_ID = base.LISTING_ID;

This measures the four main join keys at once, the listing key into the listing snapshot, the seller party key into the customer table, the browsing key, and the buyer party key. It is the source of the coverage figures used on the key diagram.

In [ ]:
WITH base AS (
  SELECT LISTING_ID, PARTY_ID, USER_PARTY_ID, NULLIF(TRIM(GA_CLIENT_ID),'') AS gacid
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
)
SELECT
  COUNT(*) AS usable_leads,
  ROUND(COUNT_IF(s.LISTING_ID IS NOT NULL)*100.0/COUNT(*),1)       AS listing_id_resolves_pct,
  ROUND(COUNT_IF(c.PARTY_ID  IS NOT NULL)*100.0/COUNT(*),1)        AS party_id_seller_resolves_pct,
  ROUND(COUNT_IF(base.gacid  IS NOT NULL)*100.0/COUNT(*),1)        AS ga_client_id_present_pct,
  ROUND(COUNT_IF(base.USER_PARTY_ID IS NOT NULL)*100.0/COUNT(*),1) AS user_party_id_present_pct
FROM base
LEFT JOIN AWS_DP_PROD.LISTINGS.LISTING_CURRENT_SNAPSHOT s ON s.LISTING_ID = base.LISTING_ID
LEFT JOIN AWS_DP_PROD.CUSTOMER.CUSTOMER_CURRENT_SNAPSHOT c ON c.PARTY_ID = base.PARTY_ID;

This is the leads to sold boats conversion by the boat location state, grouped on the listing state. Note that this particular view keeps spam leads in the count, which is how the state heat map totals were produced, so it is intentionally different from the usable lead scope elsewhere.

In [ ]:
%%sql -r dataframe_3
WITH lds AS (
  SELECT LISTING_ID, DAY, LISTING_STATE AS state
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND NULLIF(TRIM(LISTING_STATE),'') IS NOT NULL
),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS fs FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1),
f AS (SELECT lds.*, IFF(sold.fs IS NOT NULL AND sold.fs >= lds.DAY,1,0) AS conv
      FROM lds LEFT JOIN sold ON sold.LISTING_ID = lds.LISTING_ID)
SELECT state, COUNT(*) AS total_leads, SUM(conv) AS leads_on_sold, ROUND(AVG(conv)*100,1) AS rate
FROM f GROUP BY 1 HAVING COUNT(*) >= 1000 ORDER BY rate DESC;